In [1]:
# Simple Spam Message Detection Using Text Preprocessing (Ma Moh Moh Zin Min)
import pandas as pd
import re
import string
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
# 1. Dataset Loading
url = "https://raw.githubusercontent.com/mohitgupta-omg/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv"
df = pd.read_csv(url, encoding='latin-1')[['v1', 'v2']]
df.columns = ['Category', 'Message']
df['Label'] = df['Category'].map({'ham': 0, 'spam': 1})


In [3]:
# 2. NLP Preprocessing Steps
def to_lower(text):
    return text.lower()

def remove_url(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

def remove_punc(text):
    return re.sub(f"[{re.escape(string.punctuation)}]", "", text)

def remove_numbers_and_space(text):
    text = re.sub(r'\d+', '', text)
    return " ".join(text.split())

def full_preprocess(text):
    return remove_numbers_and_space(remove_punc(remove_url(to_lower(text))))

In [5]:
# 3. Model Training & Accuracy Calculation 
df['Clean_Msg'] = df['Message'].apply(full_preprocess)
tfidf = TfidfVectorizer(stop_words='english', max_features=2500)
X = tfidf.fit_transform(df['Clean_Msg'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
acc = accuracy_score(y_test, y_pred) * 100
print(f"Model Training Complete. Accuracy Score: {acc:.2f}%")

Model Training Complete. Accuracy Score: 96.32%


In [6]:
# --- Custom CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;700;900&display=swap');
.gradio-container { font-family: 'Inter', sans-serif !important; }

h1 { font-weight: 800 !important; color: #1a202c !important; font-size: 30px !important; text-align: center; }
label span { font-weight: 700 !important; color: #000000 !important; font-size: 16px !important; }

textarea, input { font-weight: 600 !important; color: #595959 !important; font-size: 16px !important; }
.gr-button-primary { background-color: #2b6cb0 !important; font-weight: 700 !important; }

.about-card { background: #f9fafb; padding: 20px; border-radius: 10px; border: 1px solid #e5e7eb; margin-bottom: 15px; }
.tech-title { color: #2b6cb0; font-weight: 700; font-size: 18px; margin-bottom: 10px; display: block; }
"""
# 4. UI logic
def nlp_pipeline_ui(message):
    s1 = to_lower(message)
    s2 = remove_url(s1)
    s3 = remove_punc(s2)
    s4 = remove_numbers_and_space(s3)
    vectorized = tfidf.transform([s4])
    pred = knn.predict(vectorized)[0]
    final_status = "🚨 This is a SPAM Message." if pred == 1 else "✅ This is a HAM Message."
    return s1, s2, s3, s4, final_status


In [16]:
# 5. UI Layout Design with About Us Page
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️Simple Spam Message Detection Using Text Preprocessing")
    
    with gr.Tabs():
        # --- Page 1: Detection Tool ---
        with gr.TabItem("🔍 Detection Tool"):
            
            with gr.Row():
                with gr.Column(scale=1):
                    msg_input = gr.Textbox(label="Input Message", lines=5, placeholder="Enter text here...")
                    submit_btn = gr.Button("Analyze Now", variant="primary")
                    
                with gr.Column(scale=2):
                    res1 = gr.Textbox(label="Step 1: Lowercasing", interactive=False)
                    res2 = gr.Textbox(label="Step 2: Links Removed", interactive=False)
                    res3 = gr.Textbox(label="Step 3: Punctuations Removed", interactive=False)
                    res4 = gr.Textbox(label="Step 4: Final Cleaned Text", interactive=False)
                    final_res = gr.Textbox(label="Final Results", interactive=False, text_align="center")
                    
            submit_btn.click(fn=nlp_pipeline_ui, inputs=msg_input, outputs=[res1, res2, res3, res4, final_res])
# --- Page 2: About Us (Technical Details) ---
        with gr.TabItem("📖 About Us"):
            with gr.Column():
                gr.Markdown("## 📋 Project Documentation & Technical Overview")
                
                with gr.Group(elem_classes="about-card"):
                    gr.Markdown("<span class='tech-title'>၁။ အသုံးပြုထားသော နည်းပညာများ (Technology Stack)</span>")
                    gr.Markdown("""
                    ဤ Project ကို Python programming language ဖြင့် ရေးသားထားပြီး အဓိကအားဖြင့် အောက်ပါ Library များကို အသုံးပြုထားပါသည်။
                    - Pandas: Dataset များကို ဖတ်ရှုရန်နှင့် Data များကို ဇယားပုံစံဖြင့် စီမံခန့်ခွဲရန်။
                    - Scikit-Learn: Machine Learning model တည်ဆောက်ရန်နှင့် TF-IDF vectorization ပြုလုပ်ရန်။
                    - Gradio: User များ အလွယ်တကူ အသုံးပြုနိုင်သော Web Interface တည်ဆောက်ရန်။
                    - Re (Regular Expression): စာသားများအတွင်းမှ မလိုအပ်သော URL များနှင့် သင်္ကေတများကို သန့်စင်ရန်။
                    """)

                with gr.Group(elem_classes="about-card"):
                    gr.Markdown("<span class='tech-title'>၂။ စာသားသန့်စင်ခြင်းလုပ်ငန်းစဉ် (NLP Preprocessing)</span>")
                    gr.Markdown("""
                    
                    Data များ၏ တိကျမှုကို ရရှိစေရန်အတွက် အောက်ပါ Preprocessing အဆင့်များကို အသုံးပြုထားပါသည်။
                    - Lowercasing: စာလုံးအကြီး/အသေး ကွဲပြားမှုကြောင့် Model အမှားမရှိစေရန် အကုန်လုံးကို စာလုံးအသေး ပြောင်းလဲခြင်း။
                    - URL Removal: Spam message များတွင် အဓိကပါဝင်သော Website Link များကို ဖယ်ရှားခြင်း။
                    - Punctuation Removal: မလိုအပ်သော သင်္ကေတများကို ဖယ်ထုတ်ပြီး စာသားကိုသာ အာရုံစိုက်စေခြင်း။
                    - Space Cleaning: ပိုနေသော Space များကို ရှင်းလင်းပြီး စာသားကို စံသတ်မှတ်ချက်အတိုင်း ပြင်ဆင်ခြင်း။
                    """)

                with gr.Group(elem_classes="about-card"):
                    gr.Markdown("<span class='tech-title'>၃။ အသုံးပြုထားသော Model (KNN Classifier)</span>")
                    gr.Markdown("""
                    
                    ဤစနစ်တွင် K-Nearest Neighbors (KNN) Algorithm ကို အသုံးပြုထားပါသည်။ ၎င်းသည် အသစ်ဝင်လာသော Message တစ်ခုကို ယခင်ရှိပြီးသား Data များနှင့် နှိုင်းယှဉ်ကာ တူညီမှုအရှိဆုံး အုပ်စု (Spam သို့မဟုတ် Ham) ကို ခွဲခြားပေးခြင်း ဖြစ်ပါသည်။ တွက်ချက်မှုများအတွက် Cosine Similarity ကို အသုံးပြုထားပါသည်။
                    """)

                with gr.Group(elem_classes="about-card"):
                    gr.Markdown("<span class='tech-title'>၄။ Vectorization (TF-IDF)</span>")
                    gr.Markdown("""
                    
                    စာသားများကို ကွန်ပျူတာမှ တွက်ချက်နိုင်သော ကိန်းဂဏန်းများအဖြစ် ပြောင်းလဲရန် TF-IDF (Term Frequency-Inverse Document Frequency) ကို အသုံးပြုထားပါသည်။ ၎င်းသည် စာလုံးများ၏ အရေးပါပုံကို မူတည်၍ အလေးပေး (Weighting) တွက်ချက်ပေးပါသည်။
                    """)

                with gr.Group(elem_classes="about-card"):
                    gr.Markdown("<span class='tech-title'>၅။ Project ၏ ရည်ရွယ်ချက်နှင့် ရှေ့ဆက်လုပ်ဆောင်မည့်အစီအစဉ်များ</span>")
                    gr.Markdown("""
                    အားသာချက်: အဆင့်ဆင့်သော Preprocessing ကြောင့် Model ၏ အဖြေမှန်ထွက်နိုင်စွမ်း မြင့်မားပါသည်။ UI မှာလည်း အဆင့်ဆင့် ပြောင်းလဲမှုကို ရှင်းလင်းစွာ မြင်တွေ့နိုင်ပါသည်။
                    
                    အကန့်အသတ်: လက်ရှိတွင် English Dataset ကိုသာ အခြေခံထားသောကြောင့် English Message များကိုသာ တိကျစွာ ခွဲခြားနိုင်ပါသေးသည်။
                    
                    ရှေ့ဆက်လုပ်ဆောင်မည့်အချက်များ: နောင်တွင် မြန်မာဘာသာစကား (Burmese Unicode/Zawgyi) များကိုပါ ခွဲခြားနိုင်စေရန် Dataset များ ထပ်မံဖြည့်စွက်ပြီး Model ကို အဆင့်မြှင့်တင်သွားရန် အစီအစဉ်ရှိပါသည်။
                    """)

if __name__ == "__main__":
    demo.launch()

C:\Users\Dell\AppData\Local\Temp\ipykernel_15772\3947550632.py:2: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
